# pycheckers package smoke demo

Small validation notebook for the package API and internal tablebase structures.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycheckers import (
    FORCED_CAPTURE,
    PROMOTION,
    QUIET,
    QuietTablebase,
    initial_position,
    inspect_game_state,
)

## Inspect One State

`inspect_game_state` returns legal quiet successors plus boundary successors grouped by metadata.

In [ ]:
info = inspect_game_state(initial_position())
{
    "state_key": info["state_key"],
    "quiet_successors": len(info["metadata_successors"][QUIET]),
    "forced_capture": info["forced_capture"],
    "promotion_possible": info["promotion_possible"],
    "terminal": info["terminal"],
}

## Generate A Small Tablebase

This uses the same dict/set-backed structures as the longer exploration notebook, but keeps the run bounded for quick validation.

In [ ]:
tablebase = QuietTablebase().run(max_rounds=3)
tablebase.summary()

## DataFrame Views

The internal structures remain Python maps and sets; these DataFrames are generated on demand for inspection and analysis.

In [ ]:
tablebase.tablebase_summary_df().T

In [ ]:
tablebase.metadata_maps_df()

In [ ]:
tablebase.rounds_df()[[
    "round",
    "frontier_in",
    "processed",
    "new_states",
    "duplicate_states",
    "new_transitions",
    "pruned_leaf_states",
    "terminal_states_total",
    "round_sec",
    "states_per_sec",
]]

## Raw Hash Maps

`transition_map(round_number, metadata)` exposes the real `source_key -> set(target_key)` maps used by the graph.

In [ ]:
quiet_round_1 = tablebase.transition_map(1, QUIET)
forced_capture_round_3 = tablebase.transition_map(3, FORCED_CAPTURE)
{
    "quiet_round_1_sources": len(quiet_round_1),
    "quiet_round_1_edges": sum(len(targets) for targets in quiet_round_1.values()),
    "forced_capture_round_3_sources": len(forced_capture_round_3),
    "forced_capture_round_3_edges": sum(len(targets) for targets in forced_capture_round_3.values()),
}

In [ ]:
tablebase.boundary_edges_df().head()

## Display A State

Pick any `state_index` from `states_df()` and pass it to `show_state`.

In [ ]:
state_index = 0
tablebase.show_state(state_index, size=1.5);